# Chess Position Clustering with DBGSOM

This notebook shows how to represent chess positions as feature vectors and cluster them with DBGSOM.

Three feature extraction approaches are demonstrated, in increasing complexity:

| Method | Dimensions | Requires | Interpretable |
|--------|-----------|----------|---------------|
| Hand-crafted (python-chess) | ~40 | `python-chess` | Yes |
| Stockfish evaluation | ~5 | Stockfish binary | Partially |
| NNUE accumulator | 2048 → 50 (PCA) | `nnue-pytorch` + `.nnue` file | No |

All three can be combined. The notebook degrades gracefully if Stockfish or nnue-pytorch are unavailable.

## Setup

In [ ]:
import sys

import chess
import chess.pgn
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from dbgsom.SomVQ import SomVQ

# Optional: Stockfish
try:
    import chess.engine

    STOCKFISH_PATH = "stockfish"  # adjust if needed, e.g. r"C:\stockfish\stockfish.exe"
    _engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
    _engine.quit()
    HAS_STOCKFISH = True
    print("Stockfish: available")
except Exception:
    HAS_STOCKFISH = False
    print("Stockfish: not found — skipping engine features")

# Optional: nnue-pytorch
NNUE_PYTORCH_PATH = "../nnue-pytorch"  # adjust to your clone
NNUE_FILE_PATH = "nn-xxxxxxxx.nnue"  # adjust to your .nnue file
try:
    sys.path.insert(0, NNUE_PYTORCH_PATH)
    import data_loader
    import model as NNUEModule
    import torch

    HAS_NNUE = True
    print("nnue-pytorch: available")
except ImportError:
    HAS_NNUE = False
    print("nnue-pytorch: not found — skipping NNUE embeddings")

## Sample Data — Opening Positions

We generate positions from eight well-known opening families by replaying move sequences.
Each position in the sequence is stored as a FEN string along with its opening label.

Replace `OPENING_LINES` with positions loaded from your Lichess PGN export for real data.

In [ ]:
OPENING_LINES = {
    "Ruy Lopez": "e2e4 e7e5 g1f3 b8c6 f1b5 a7a6 b5a4 g8f6 e1g1 f8e7 f1e1 b7b5 a4b3 d7d6 c2c3 e8g8",
    "Sicilian Najdorf": "e2e4 c7c5 g1f3 d7d6 d2d4 c5d4 f3d4 g8f6 b1c3 a7a6 c1g5 e7e6 d1d2 f8e7 e1c1 e8g8",
    "French Defense": "e2e4 e7e6 d2d4 d7d5 b1c3 g8f6 c1g5 f8e7 e4e5 f6d7 g5e7 d8e7 d1d2 a7a6 f2f4 c7c5",
    "Caro-Kann": "e2e4 c7c6 d2d4 d7d5 b1c3 d5e4 c3e4 g8f6 e4f6 e7f6 c2c3 f8d6 g1f3 e8g8 f1d3 b8d7",
    "Queens Gambit": "d2d4 d7d5 c2c4 e7e6 b1c3 g8f6 c1g5 f8e7 e2e3 e8g8 g1f3 b8d7 d1c2 c7c6 f1d3 d5c4",
    "Kings Indian": "d2d4 g8f6 c2c4 g7g6 b1c3 f8g7 e2e4 d7d6 g1f3 e8g8 f1e2 e7e5 e1g1 b8c6 d4d5 c6e7",
    "English Opening": "c2c4 e7e5 b1c3 g8f6 g1f3 b8c6 e2e3 f8b4 d1c2 e8g8 a2a3 b4c3 c2c3 d7d5 c4d5 f6d5",
    "Kings Gambit": "e2e4 e7e5 f2f4 e5f4 g1f3 g7g5 h2h4 g5g4 f3e5 g8f6 f1c4 d7d5 e4d5 f8d6 e1g1 d6e5",
}


def positions_from_line(uci_moves: str) -> list[chess.Board]:
    board = chess.Board()
    boards = []
    for uci in uci_moves.split():
        board.push(chess.Move.from_uci(uci))
        if not board.is_check():  # Stockfish can't eval positions in check
            boards.append(board.copy())
    return boards


all_boards, all_labels = [], []
for name, line in OPENING_LINES.items():
    boards = positions_from_line(line)
    all_boards.extend(boards)
    all_labels.extend([name] * len(boards))

all_fens = [b.fen() for b in all_boards]
label_names = list(OPENING_LINES.keys())
label_ids = np.array([label_names.index(l) for l in all_labels])

print(f"{len(all_fens)} positions from {len(label_names)} opening families")

## Feature Extraction

### Method 1 — Hand-crafted Board Features

Encodes structural properties of the position that are directly readable from the FEN:

- **Material** (12): piece count per type and color
- **Mobility** (2): legal move count for each side
- **Pawn structure** (4): doubled and isolated pawns per color
- **Castling rights** (4): binary flags
- **Side to move** (1): +1 white, −1 black

In [ ]:
def board_features(board: chess.Board) -> np.ndarray:
    features = []

    # Material
    for color in [chess.WHITE, chess.BLACK]:
        for pt in [
            chess.PAWN,
            chess.KNIGHT,
            chess.BISHOP,
            chess.ROOK,
            chess.QUEEN,
            chess.KING,
        ]:
            features.append(len(board.pieces(pt, color)))

    # Mobility — legal moves for both sides
    features.append(board.legal_moves.count())
    board.push(chess.Move.null())
    features.append(board.legal_moves.count())
    board.pop()

    # Pawn structure
    for color in [chess.WHITE, chess.BLACK]:
        pawns = board.pieces(chess.PAWN, color)
        files = [chess.square_file(sq) for sq in pawns]
        doubled = sum(files.count(f) > 1 for f in set(files))
        isolated = sum((f - 1) not in files and (f + 1) not in files for f in files)
        features += [doubled, isolated]

    # Castling rights
    features += [
        int(board.has_kingside_castling_rights(chess.WHITE)),
        int(board.has_queenside_castling_rights(chess.WHITE)),
        int(board.has_kingside_castling_rights(chess.BLACK)),
        int(board.has_queenside_castling_rights(chess.BLACK)),
    ]

    # Side to move
    features.append(1.0 if board.turn == chess.WHITE else -1.0)

    return np.array(features, dtype=np.float32)


X_hand = np.array([board_features(b) for b in all_boards])
print(f"Hand-crafted features: {X_hand.shape}")

### Method 2 — Stockfish Evaluation Features

Queries Stockfish at `depth=12` per position and extracts:

- **Centipawn score** (relative to side to move)
- **WDL probabilities** (win / draw / loss) from the neural-network-based estimate
- **Sharpness**: score gap between best and second-best move (requires `multipv=2`)

Skipped automatically if Stockfish is not found on `PATH`.

In [ ]:
def stockfish_features_batch(
    boards: list[chess.Board],
    engine_path: str,
    depth: int = 12,
) -> np.ndarray:
    results = []
    with chess.engine.SimpleEngine.popen_uci(engine_path) as engine:
        engine.configure({"UCI_ShowWDL": True})
        for board in boards:
            info = engine.analyse(board, chess.engine.Limit(depth=depth), multipv=2)

            score = info[0]["score"].relative
            cp = score.score(mate_score=10_000) / 100.0

            wdl = info[0].get("wdl")
            w = wdl.wins / 1000.0 if wdl else 0.5
            d = wdl.draws / 1000.0 if wdl else 0.0
            l = wdl.losses / 1000.0 if wdl else 0.5

            sharpness = 0.0
            if len(info) >= 2:
                s0 = info[0]["score"].relative.score(mate_score=10_000)
                s1 = info[1]["score"].relative.score(mate_score=10_000)
                sharpness = abs(s0 - s1) / 100.0

            results.append([cp, w, d, l, sharpness])
    return np.array(results, dtype=np.float32)


if HAS_STOCKFISH:
    X_sf = stockfish_features_batch(all_boards, STOCKFISH_PATH)
    print(f"Stockfish features: {X_sf.shape}")
else:
    X_sf = np.zeros((len(all_boards), 5), dtype=np.float32)
    print("Stockfish not available — placeholder zeros used")

### Method 3 — NNUE Accumulator Embedding

Loads a Stockfish `.nnue` weight file via `nnue-pytorch` and extracts the **accumulator layer** — the first hidden layer after the HalfKA feature transformer.

Architecture (current Stockfish HalfKA_v2_hm):
```
HalfKA features (~41k binary, sparse)
    → Accumulator: 1024 neurons × 2 king-perspectives  =  2048 dim
    → ClippedReLU
    → Linear 32 → Linear 32 → 1 scalar (centipawn eval)
```

The 2048-dim accumulator is reduced to 50 dimensions with PCA before feeding into the SOM.

**Requirements:**
```bash
git clone https://github.com/official-stockfish/nnue-pytorch
pip install -r nnue-pytorch/requirements.txt
# Download a .nnue file from https://tests.stockfishchess.org/nns
```

The `cross_check_eval.py` script from the repo provides `data_loader.get_sparse_batch_from_fens` —
the key function that handles HalfKA feature generation from FEN strings directly.

In [ ]:
def load_nnue_model(nnue_path: str, device: str = "cpu"):
    config = NNUEModule.NNUELightningConfig()
    with open(nnue_path, "rb") as f:
        reader = NNUEModule.NNUEReader(f, config.features, config.model_config)
    model = reader.model
    model.to(device)
    model.eval()
    return model


def nnue_embeddings(
    fens: list[str],
    model,
    device: str = "cpu",
    batch_size: int = 256,
) -> np.ndarray:
    """
    Returns the NNUE accumulator for each FEN as a 2048-dim vector.
    Uses data_loader.get_sparse_batch_from_fens from nnue-pytorch.
    """
    all_embeddings = []
    feature_name = model.input_feature_name

    # Collect intermediate accumulator via forward hook
    accumulator_outputs = []

    def _hook(module, inp, out):
        accumulator_outputs.append(out.detach().cpu())

    # Attach hook to the input (feature transformer) layer
    # The attribute name varies by nnue-pytorch version — check model.named_modules()
    ft_layer = None
    for name, module in model.named_modules():
        if "input" in name.lower() and hasattr(module, "weight"):
            ft_layer = module
            break
    if ft_layer is None:
        raise RuntimeError(
            "Could not find feature transformer layer. "
            "Inspect model.named_modules() and set ft_layer manually."
        )

    handle = ft_layer.register_forward_hook(_hook)

    for start in range(0, len(fens), batch_size):
        batch_fens = fens[start : start + batch_size]
        b = data_loader.get_sparse_batch_from_fens(
            feature_name,
            batch_fens,
            [0] * len(batch_fens),  # score placeholder
            [1] * len(batch_fens),  # result placeholder
            [0] * len(batch_fens),  # ply placeholder
        )
        (
            us,
            them,
            white_indices,
            white_values,
            black_indices,
            black_values,
            _,
            _,
            psqt_indices,
            layer_stack_indices,
        ) = b.contents.get_tensors(device)

        accumulator_outputs.clear()
        with torch.no_grad():
            _ = model.forward(
                us,
                them,
                white_indices,
                white_values,
                black_indices,
                black_values,
                psqt_indices,
                layer_stack_indices,
            )
        data_loader.destroy_sparse_batch(b)

        # Hook captures both white and black perspectives separately;
        # concatenate them to form the 2048-dim embedding.
        if len(accumulator_outputs) >= 2:
            emb = torch.cat(accumulator_outputs[:2], dim=1).numpy()
        else:
            emb = accumulator_outputs[0].numpy()
        all_embeddings.append(emb)

    handle.remove()
    return np.vstack(all_embeddings)


if HAS_NNUE:
    nnue_model = load_nnue_model(NNUE_FILE_PATH)
    X_nnue_raw = nnue_embeddings(all_fens, nnue_model)
    print(f"NNUE raw embeddings: {X_nnue_raw.shape}")

    # Reduce to 50 dims with PCA
    pca_nnue = PCA(n_components=50, random_state=42)
    X_nnue = pca_nnue.fit_transform(X_nnue_raw)
    print(
        f"NNUE after PCA: {X_nnue.shape}  "
        f"(explained variance: {pca_nnue.explained_variance_ratio_.sum():.1%})"
    )
else:
    X_nnue = np.zeros((len(all_boards), 50), dtype=np.float32)
    print("nnue-pytorch not available — placeholder zeros used")

## Feature Combination & Normalization

All three feature blocks are concatenated and standardized.
When Stockfish or nnue-pytorch are unavailable, the corresponding block is all-zeros and has no effect after `StandardScaler` (zero variance → zero contribution after scaling).

In [ ]:
# Use only available feature blocks (drop zero-variance columns)
blocks = [X_hand]
if HAS_STOCKFISH:
    blocks.append(X_sf)
if HAS_NNUE:
    blocks.append(X_nnue)

X_raw = np.hstack(blocks)

# Remove zero-variance features before scaling
nonzero_var = X_raw.var(axis=0) > 0
X_raw = X_raw[:, nonzero_var]

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

feature_sources = (
    f"hand-crafted({X_hand.shape[1]})"
    + (f" + stockfish({X_sf.shape[1]})" if HAS_STOCKFISH else "")
    + (f" + NNUE-PCA({X_nnue.shape[1]})" if HAS_NNUE else "")
)
print(f"Final feature matrix: {X.shape}  [{feature_sources}]")

## DBGSOM Training

DBGSOM grows dynamically — no grid size needed upfront.
With only ~120 positions, a small `max_neurons` is sufficient.

In [ ]:
import time

som = SomVQ(
    n_iter=300,
    lambda_=5.0,
    max_neurons=40,
    decay_function="linear",
    sigma_end=0.5,
    random_state=42,
    verbose=True,
)

t0 = time.perf_counter()
som.fit(X)
elapsed = time.perf_counter() - t0

print(
    f"\nNeurons: {len(som.neurons_)}  |  "
    f"QE: {som.calculate_quantization_error(X):.4f}  |  "
    f"TE: {som.topographic_error_:.4f}  |  "
    f"Time: {elapsed:.2f}s"
)

## Visualization

Two plots:

1. **SOM grid (PCA projection)** — neuron positions in the 2D data space,
   colored by the dominant opening family assigned to each neuron.
2. **Hit map** — which neurons are most frequently the BMU across all positions.

In [ ]:
COLORS = plt.cm.tab10(np.linspace(0, 1, len(label_names)))
COLOR_MAP = dict(zip(label_names, COLORS))

# Project weights and data into 2D PCA space
pca_vis = PCA(n_components=2, random_state=42)
pca_vis.fit(X)
weights_2d = pca_vis.transform(som.weights_)
data_2d = pca_vis.transform(X)

# Assign dominant opening label to each neuron
bmu_per_sample = np.array([som._best_matching_unit(x) for x in X])
neuron_list = list(som.neurons_)

neuron_label = {}
neuron_hits = {n: 0 for n in neuron_list}
for sample_idx, bmu in enumerate(bmu_per_sample):
    neuron_hits[bmu] = neuron_hits.get(bmu, 0) + 1

for neuron in neuron_list:
    assigned = [all_labels[i] for i, b in enumerate(bmu_per_sample) if b == neuron]
    if assigned:
        neuron_label[neuron] = max(set(assigned), key=assigned.count)
    else:
        neuron_label[neuron] = None

node_pos = dict(zip(neuron_list, weights_2d))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Plot 1: Opening label map ---
ax = axes[0]
ax.set_title("SOM — dominant opening per neuron", fontsize=12)

for u, v in som.som_.edges():
    x0, y0 = node_pos[u]
    x1, y1 = node_pos[v]
    ax.plot([x0, x1], [y0, y1], color="lightgray", lw=0.8, zorder=1)

for neuron, pos in node_pos.items():
    label = neuron_label.get(neuron)
    color = COLOR_MAP[label] if label else "white"
    ax.scatter(*pos, color=color, s=200, zorder=3, edgecolors="gray", linewidths=0.5)

# Scatter raw data points (small, semi-transparent)
for i, (pos, label) in enumerate(zip(data_2d, all_labels)):
    ax.scatter(*pos, color=COLOR_MAP[label], s=15, alpha=0.4, zorder=2)

legend_patches = [mpatches.Patch(color=COLOR_MAP[n], label=n) for n in label_names]
ax.legend(handles=legend_patches, fontsize=8, loc="best")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")

# --- Plot 2: Hit map ---
ax = axes[1]
ax.set_title("SOM — hit count per neuron", fontsize=12)

for u, v in som.som_.edges():
    x0, y0 = node_pos[u]
    x1, y1 = node_pos[v]
    ax.plot([x0, x1], [y0, y1], color="lightgray", lw=0.8, zorder=1)

hits = np.array([neuron_hits.get(n, 0) for n in neuron_list], dtype=float)
sc = ax.scatter(
    weights_2d[:, 0],
    weights_2d[:, 1],
    c=hits,
    cmap="YlOrRd",
    s=200,
    zorder=3,
    edgecolors="gray",
    linewidths=0.5,
)
plt.colorbar(sc, ax=ax, label="Hit count")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")

plt.tight_layout()
plt.show()

## Cluster Analysis — Neuron Summary

For each neuron: which openings are assigned, how many hits, and what the dominant opening is.

In [ ]:
import pandas as pd

rows = []
for neuron in neuron_list:
    assigned = [all_labels[i] for i, b in enumerate(bmu_per_sample) if b == neuron]
    if not assigned:
        continue
    dominant = max(set(assigned), key=assigned.count)
    purity = assigned.count(dominant) / len(assigned)
    rows.append(
        {
            "Neuron": neuron,
            "Hits": len(assigned),
            "Dominant opening": dominant,
            "Purity": f"{purity:.0%}",
            "All openings": ", ".join(sorted(set(assigned))),
        }
    )

df = pd.DataFrame(rows).sort_values("Hits", ascending=False).reset_index(drop=True)
df

## Optional — Inspect Positions Mapped to a Neuron

Select a neuron and print the FENs assigned to it — useful for manual inspection.

In [ ]:
# Change this to any neuron id from the table above
INSPECT_NEURON = neuron_list[0]

print(f"Positions assigned to neuron {INSPECT_NEURON}:")
for i, bmu in enumerate(bmu_per_sample):
    if bmu == INSPECT_NEURON:
        print(f"  [{all_labels[i]:20s}]  {all_fens[i]}")

## Scaling to Lichess Data

Replace the sample data with real positions from your Lichess PGN export:

In [ ]:
def load_fens_from_pgn(
    pgn_path: str,
    ply_range: tuple[int, int] = (10, 20),
    max_games: int = 5_000,
) -> list[str]:
    """Extract one FEN per game at a random ply within ply_range."""
    import random

    fens = []
    with open(pgn_path) as f:
        while len(fens) < max_games:
            game = chess.pgn.read_game(f)
            if game is None:
                break
            board = game.board()
            moves = list(game.mainline_moves())
            target_ply = random.randint(*ply_range)
            for move in moves[:target_ply]:
                board.push(move)
            if not board.is_check():
                fens.append(board.fen())
    return fens


# Example usage (uncomment and adjust path):
# fens_lichess = load_fens_from_pgn("lichess_games.pgn", ply_range=(10, 20), max_games=10_000)
# X_lichess = np.array([board_features(chess.Board(fen)) for fen in fens_lichess])
# X_lichess_scaled = StandardScaler().fit_transform(X_lichess)
#
# som_big = SomVQ(n_iter=500, lambda_=50.0, max_neurons=200, random_state=42)
# som_big.fit(X_lichess_scaled)
print("Load your PGN here — see comments above.")